# Lista 01 - CKP8366 - TÓPICOS ESPECIAIS EM LÓGICA I

Realizado por Gustavo Fernandez Vidal Vazquez, matrícula 603150. 

In [4]:
from pysat.formula import *

## Tarefa 1 - Escreva a função (recursiva) subFormula(f) que retorna todas as subfórmulas de uma fórmula.
Dica: podemos saber se uma fórmula f é atômica (do tipo Atom) usando a
função booleana isinstance(f,Atom), para saber se uma fórmula é do tipo And, podemos
usar isinstance(f, And), e assim por diante . . . )

### Função Auxiliar

In [5]:
# Função auxiliar para detectar caso padrão
def isInstanceOfAndOrEqualsXOr(f) :
    return (isinstance(f, And) or
            isinstance(f, Or) or
            isinstance(f, Equals) or
            isinstance(f, XOr))

### Função Principal

In [97]:
def subFormula(f):
    # Não considerei átomo como fórmula
    if isinstance(f, Atom):
        return []
    # Caso padrão: retorna fórmula atual + chamada recursiva para subfórmulas maximais
    if isInstanceOfAndOrEqualsXOr(f):
        return [f] + sum([subFormula(sub) for sub in f.subformulas], [])
    # Caso neg: apenas uma subfórmula
    if isinstance(f, Neg):
        return [f] + subFormula(f.subformula)
    # Caso implies: acesso das subfórmulas via .left e .right
    if isinstance(f, Implies):
        return [f] + subFormula(f.left) + subFormula(f.right)

### Exemplo Teste

In [98]:
lista = []
n = 3
for j in range(n):
    for k in range(j + 1, n):
        lista.append(Neg(And(Atom('p1,' + str(j)), Atom('p1,' + str(k)))))

f = And(*lista)

In [99]:
for sub in subFormula(f):
    print(sub)

~(p1,0 & p1,1) & ~(p1,0 & p1,2) & ~(p1,1 & p1,2)
~(p1,0 & p1,1)
p1,0 & p1,1
~(p1,0 & p1,2)
p1,0 & p1,2
~(p1,1 & p1,2)
p1,1 & p1,2


# Tarefa 2 - Escreva a função (recursiva) formRand(n) que gera uma fórmula recursiva com n nós.

In [100]:
import random

def chooseRandomAtom(k) :
    return (Atom('p' + str(random.randint(1, k))))

operators = ['AND', 'OR', 'NEG']

# Adicionei um k opcional para determinar a pool de literais da fórmula. Por padrão, k = n
def formRand(n, k = None):
    if k is None:
        k = n
        
    # Fórmula com um nó só pode ser literal
    if (n == 1): 
        return chooseRandomAtom(k)
        
    # Fórmula com dois nós só pode ser negação
    if (n == 2):
        return Neg(formRand(n-1, k))

    # Para três ou mais nós, podemos aleatorizar o operador
    op = random.choice(operators)
    if (op == 'NEG'):
        return Neg(formRand(n-1, k))

    # Tenho (n - 1) nós disponíveis para dividir na subárvore esquerda e direita.

    # Tamanho esquerdo tem no máximo (n - 2) nós para garantir que árvore direita possui pelo menos um nó.
    left_size = random.randint(1, n - 2)
    # subárvore direita possui nós que sobraram
    right_size = (n - 1) - left_size

    # Para adicionar mas fórmulas (XOr, Implies, ...) seria só aumentar a lista de operadores 
    # e adicionar um if para cada caso aqui. 
    if (op == 'AND'):
        return And(formRand(left_size, k), formRand(right_size, k))             
    if (op == 'OR'):
        return Or(formRand(left_size, k), formRand(right_size, k))    

In [101]:
print(formRand(10, 3))

~(~p2 | ((p2 | p1) & ~p1))


# Tarefa 3 - Escreva a função fncon(f) que, dada uma fórmula, produz outra na forma normal conjuntiva (um objeto do tipo And cujas subfórmulas são do tipo Or que por sua vez possui apenas literais como subformulas).

Pequenos detalhes:
- Uso do "t_" + counter não é o melhor, já que pode ter colisão com nome na fórmula original.
- O ideal aqui seria atribuir os números para as cláusulas diretamente, assim seria fácil construir um átomo. Mas como a tarefa de conversão para lista de inteiros é a próxima, fiz aqui uma simplificação para construir a CNF ainda nomeada (sem representação de inteiros)

In [102]:
# Implementação da transformação de Tseytin
# https://en.wikipedia.org/wiki/Tseytin_transformation

# 1. Percorrer fórmula
# 2. Adicionar equivalências e substituir na fórmula original (aqui eu uso um mapa)
# 3. Trocar equivalências usando regras de substituição para CNF

def fncon(f):
    mapping = {}
    clauses = []
    counter = 0

    def new_atom():
        nonlocal counter
        counter += 1
        return Atom(("t_" + str(counter)))

    def encode(f):

        # Se temos alguma expressão que já tem nome no mapa (já temos a equivalência (...) <-> t
        # simplesmente trocamos (...) por t
        if f in mapping:
            return mapping[f]

        # Átomos não mudam
        if isinstance(f, Atom):
            return f

        # Expressão não é atômica e não possui equivalência : criamos novo átomo.
        z = new_atom()
        mapping[f] = z

        # Regra de equivalência caso a fórmula seja um And de aridade qualquer
        # z <-> (a & b & c)
        # (~z | a) ( ~z | b) (~z | c ) & (~a | ~b | ~c | z)
        if isinstance(f, And):
            children = [encode(sub) for sub in f.subformulas]

            for child in children:
                clauses.append(Or(Neg(z), child))

            clauses.append(Or(*([Neg(a) for a in children] + [z])))

        # Regra de equivalência caso a fórmula seja um Or de aridade qualquer
        # z <-> (a | b | c)
        # (z | ~a) (z | ~b) (z | ~c ) & (a | b | c | ~z)
        elif isinstance(f, Or):
            children = [encode(sub) for sub in f.subformulas]

            for child in children:
                clauses.append(Or(Neg(child), z))

            clauses.append(Or(Neg(z), *children))

        # Regra de equivalência caso a fórmula seja uma negação
        # z <-> ~a 
        # (~z | ~a) & (z | a)
        elif isinstance(f, Neg):
            a = encode(f.subformula)

            clauses.append(Or(Neg(z), Neg(a)))
            clauses.append(Or(z, a))

        return z

    # Chamamos encode pra raiz e adicionamos a raiz no conjunto de cláusulas
    root = encode(f)
    clauses.append(root)

    return And(*clauses)

In [103]:
# ~(a | b)
f = Neg(Or(Atom("a"), Atom("b")))

In [104]:
new_f = fncon(f)
print(new_f)

(~a | t_2) & (~b | t_2) & (~t_2 | a | b) & (~t_1 | ~t_2) & (t_1 | t_2) & t_1


# Tarefa 4 - Escreva a função clausulas(f) que retorna uma lista de listas de inteiros representando as cláusulas da fórmula obtida na tarefa anterior na forma normal conjuntiva

In [105]:
from pysat.formula import IDPool

def clausulas(f):
    idp = IDPool()
    clauses = []


    # id se for átomo e - id se for neg
    def lit_to_int(lit):
        if isinstance(lit, Atom):
            return idp.obj2id[lit]
        elif isinstance(lit, Neg):
            return -idp.obj2id[lit.subformula]

    if isinstance(f, And):
        for clause in f.subformulas:
            if isinstance(clause, Or):
                c = []
                for lit in clause.subformulas:
                    c.append(lit_to_int(lit))
                clauses.append(c)
            else:
                # unitária
                clauses.append([lit_to_int(clause)])
    else:
        # caso f já seja uma única cláusula
        if isinstance(f, Or):
            c = [lit_to_int(l) for l in f.subformulas]
            clauses.append(c)
        else:
            clauses.append([lit_to_int(f)])

    return clauses

In [106]:
clausulas(new_f)

[[-1, 2], [-3, 2], [-2, 1, 3], [-4, -2], [4, 2], [4]]